# Global Solution 2026 — SpaceTrain Energy

Este notebook executa uma prova de conceito simples para estimar o custo energético do treinamento de uma rede neural simples aplicada a um cenário espacial ou Terra-espaço.

Atenção: o foco não é obter o melhor modelo possível. O foco é observar como número de variáveis, quantidade de épocas e escala de dados aumentam acessos à RAM, registradores, operações da ULA e energia estimada.

## Passo 1 — Enviar o dataset

1. No Colab, clique no ícone de pasta à esquerda.
2. Clique em **Upload**.
3. Envie o arquivo `dataset_spacetrain_energia_5h.csv`.
4. Depois rode a célula abaixo.

In [ ]:
import numpy as np
import pandas as pd

arquivo = "dataset_spacetrain_energia_5h.csv"
dados = pd.read_csv(arquivo)

display(dados.head())
print("Total de linhas:", len(dados))
print("Colunas:", list(dados.columns))
print("Quantidade de registros com risco crítico:", int(dados["risco_critico"].sum()))

## Passo 2 — Conferir o significado das colunas

- `minuto`: instante do monitoramento simulado.
- `temperatura_equipamento_c`: temperatura do equipamento ou módulo monitorado.
- `radiacao_relativa`: intensidade relativa de radiação no ambiente.
- `perda_comunicacao_pct`: percentual de perda de comunicação.
- `variacao_energia_pct`: variação percentual no fornecimento de energia.
- `atraso_sinal_ms`: atraso no sinal de comunicação.
- `indice_poeira`: indicador didático de poeira ou partículas no ambiente.
- `risco_critico`: classe que a rede tenta aprender. Valor 0 significa normal; valor 1 significa risco crítico.

## Passo 3 — Definir os jobs de treinamento

A equipe deve executar os três jobs já definidos. Não é necessário alterar o dataset. Os jobs diferem pelo número de variáveis usadas e pela quantidade de épocas.

In [ ]:
jobs = {
    "A - Edge espacial simples": {
        "features": ["temperatura_equipamento_c", "radiacao_relativa"],
        "epocas": 8,
        "contexto": "sensor local com poucas variáveis"
    },
    "B - Operação Terra-espaço": {
        "features": ["temperatura_equipamento_c", "radiacao_relativa", "perda_comunicacao_pct", "atraso_sinal_ms"],
        "epocas": 15,
        "contexto": "operação com telemetria e comunicação"
    },
    "C - Escala ampliada": {
        "features": ["temperatura_equipamento_c", "radiacao_relativa", "perda_comunicacao_pct", "variacao_energia_pct", "atraso_sinal_ms", "indice_poeira"],
        "epocas": 25,
        "contexto": "cenário com mais sensores e maior repetição"
    }
}

jobs

## Passo 4 — Funções do simulador

A célula abaixo cria uma rede neural simples com uma saída do tipo sigmoide. O simulador contabiliza, de forma didática, acessos à RAM, acessos a registradores e operações da ULA durante o treinamento.

Premissa energética didática:

- RAM = 640 pJ por acesso.
- Registradores = 5 pJ por acesso.
- ULA = 10 pJ por operação.

Fórmula:

`Energia (pJ) = 640 x RAM + 5 x Registradores + 10 x ULA`

In [ ]:
def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1 / (1 + np.exp(-z))


def treinar_rede_simples(df, features, target="risco_critico", epocas=10, taxa_aprendizado=0.08):
    X = df[features].values.astype(float)
    y = df[target].values.astype(float)

    # Separação simples entre treino e teste
    rng = np.random.default_rng(123)
    indices = np.arange(len(df))
    rng.shuffle(indices)
    tamanho_treino = int(0.7 * len(df))
    idx_treino = indices[:tamanho_treino]
    idx_teste = indices[tamanho_treino:]

    X_treino, y_treino = X[idx_treino], y[idx_treino]
    X_teste, y_teste = X[idx_teste], y[idx_teste]

    # Normalização simples
    media = X_treino.mean(axis=0)
    desvio = X_treino.std(axis=0)
    desvio[desvio == 0] = 1
    X_treino = (X_treino - media) / desvio
    X_teste = (X_teste - media) / desvio

    n_features = X_treino.shape[1]
    pesos = np.zeros(n_features)
    bias = 0.0

    RAM = 0
    Registradores = 0
    ULA = 0
    forwards = 0
    atualizacoes = 0

    for epoca in range(epocas):
        for xi, yi in zip(X_treino, y_treino):
            # Forward
            RAM += n_features              # leitura das features
            RAM += n_features              # leitura dos pesos
            RAM += 1                       # leitura do bias
            Registradores += 2*n_features + 2
            ULA += n_features              # multiplicações xi*w
            ULA += max(n_features - 1, 1)  # somas parciais
            ULA += 1                       # soma do bias
            ULA += 4                       # aproximação didática da sigmoid

            z = np.dot(xi, pesos) + bias
            prob = sigmoid(z)
            forwards += 1

            # Atualização dos pesos
            RAM += 1                       # leitura do alvo/target
            Registradores += n_features + 4
            ULA += 1                       # erro = prob - alvo
            erro = prob - yi

            grad_pesos = erro * xi
            grad_bias = erro
            ULA += 2*n_features + 2        # gradiente e atualização
            RAM += n_features + 1          # escrita dos pesos e bias

            pesos -= taxa_aprendizado * grad_pesos
            bias -= taxa_aprendizado * grad_bias
            atualizacoes += 1

    probabilidades = sigmoid(np.dot(X_teste, pesos) + bias)
    predicoes = (probabilidades >= 0.5).astype(int)
    acuracia = (predicoes == y_teste).mean()

    energia_pJ = 640*RAM + 5*Registradores + 10*ULA

    return {
        "n_amostras_treino": len(X_treino),
        "n_amostras_teste": len(X_teste),
        "n_features": n_features,
        "epocas": epocas,
        "forwards": forwards,
        "atualizacoes": atualizacoes,
        "RAM": RAM,
        "Registradores": Registradores,
        "ULA": ULA,
        "energia_treino_pJ": energia_pJ,
        "energia_treino_J": energia_pJ * 1e-12,
        "acuracia_teste": acuracia
    }


def classificar_infraestrutura(energia_pJ):
    if energia_pJ <= 1e10:
        return "Máquina local"
    elif energia_pJ <= 1e12:
        return "Workstation/GPU"
    elif energia_pJ <= 1e14:
        return "Cluster"
    else:
        return "Supercomputador ou revisão arquitetural/neuromórfica"

## Passo 5 — Rodar os três jobs

Rode a célula abaixo e copie a tabela gerada para o relatório da equipe.

In [ ]:
resultados = []

for nome_job, config in jobs.items():
    saida = treinar_rede_simples(
        dados,
        features=config["features"],
        epocas=config["epocas"]
    )
    saida["job"] = nome_job
    saida["contexto"] = config["contexto"]
    resultados.append(saida)

resultado_jobs = pd.DataFrame(resultados)
resultado_jobs = resultado_jobs[[
    "job", "contexto", "n_amostras_treino", "n_features", "epocas",
    "forwards", "atualizacoes", "RAM", "Registradores", "ULA",
    "energia_treino_pJ", "energia_treino_J", "acuracia_teste"
]]

display(resultado_jobs)

## Passo 6 — Projetar a energia em escala

Agora o notebook estima quanto custaria treinar em escalas maiores. A estimativa usa o custo médio por amostra e por época observado em cada job.

In [ ]:
escalas = [
    {"escala": "Protótipo didático", "amostras": 10_000, "epocas": 20},
    {"escala": "Operação regional", "amostras": 200_000, "epocas": 50},
    {"escala": "Constelação global", "amostras": 10_000_000, "epocas": 100},
    {"escala": "Missão massiva", "amostras": 100_000_000, "epocas": 200},
]

projecoes = []

for _, linha in resultado_jobs.iterrows():
    energia_por_amostra_epoca = linha["energia_treino_pJ"] / (linha["n_amostras_treino"] * linha["epocas"])
    for escala in escalas:
        energia_estimada = energia_por_amostra_epoca * escala["amostras"] * escala["epocas"]
        projecoes.append({
            "job": linha["job"],
            "escala": escala["escala"],
            "amostras": escala["amostras"],
            "epocas": escala["epocas"],
            "energia_estimada_pJ": energia_estimada,
            "energia_estimada_J": energia_estimada * 1e-12,
            "decisao_infraestrutura": classificar_infraestrutura(energia_estimada)
        })

resultado_escala = pd.DataFrame(projecoes)
display(resultado_escala)

## Passo 7 — Resumo automático para copiar no relatório

Copie o resumo gerado abaixo e use como evidência de execução. Depois escreva a interpretação da equipe com suas próprias palavras.

In [ ]:
print("RESUMO PARA O RELATÓRIO")
print("="*60)
for _, linha in resultado_jobs.iterrows():
    print(f"{linha['job']}: features={linha['n_features']}, épocas={linha['epocas']}, RAM={linha['RAM']}, Registradores={linha['Registradores']}, ULA={linha['ULA']}, energia={linha['energia_treino_pJ']:.0f} pJ, acurácia={linha['acuracia_teste']:.3f}")

print("\nProjeção de infraestrutura:")
for _, linha in resultado_escala.iterrows():
    print(f"{linha['job']} | {linha['escala']} | energia={linha['energia_estimada_pJ']:.2e} pJ | {linha['decisao_infraestrutura']}")
